# CO-SPY Fusion v3 — Full SDXL Evaluation

Eval **CO-SPY Fusion v3 = Semantic + Artifact + NPR → Fusion v3**
trên SDXL của Co-Spy-Bench.

Fake data được đọc từ 5 Kaggle Input riêng:

- `SDXL_mscoco`
- `SDXL_flickr`
- `SDXL_cc3m`
- `SDXL_textcaps`
- `SDXL_sbu`

Mỗi input có cấu trúc:

```text
SDXL-full/
└── synthetic/
    └── <source>/
        └── stabilityai@stable-diffusion-xl-base-1.0/
```

Real benchmark:
- MSCOCO: lấy từ input `Co-Spy_v1`
- CC3M / SBU / TextCaps: official `real_image_examples`
- Flickr: Flickr30k test Parquet

Mặc định mỗi source eval:
- 2.000 real
- 2.000 fake

Tổng: **20.000 ảnh**.

Notebook giữ nguyên các fix đã dùng cho PG-v2-512:
- load raw full Fusion v3 `state_dict`
- kiểm tra missing/unexpected keys
- patch artifact features `(B,V,F)`
- preflight 1 batch trước full eval


In [1]:
from pathlib import Path
import os, sys, subprocess, time, json, csv, gc, inspect, zipfile
import requests

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "cospy"

BATCH = 16
NUM_PER_CLASS = 2000
JPEG_QUALITY = 95
ARTIFACT_VIEWS = 1

SOURCES = ["mscoco", "flickr", "cc3m", "textcaps", "sbu"]
MODEL_NAME = "stabilityai@stable-diffusion-xl-base-1.0"

OUT_DIR = WORK / "eval_sdxl_fusionv3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT   :", INPUT)
print("MODEL   :", MODEL_NAME)
print("Sources :", SOURCES)
print("N/class :", NUM_PER_CLASS)
print("Total eval images:", NUM_PER_CLASS * 2 * len(SOURCES))


INPUT   : /kaggle/input
MODEL   : stabilityai@stable-diffusion-xl-base-1.0
Sources : ['mscoco', 'flickr', 'cc3m', 'textcaps', 'sbu']
N/class : 2000
Total eval images: 20000


In [2]:
# SETUP REPO + DEPENDENCIES
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/datnguyentrung/CO-SPY.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open-clip-torch==2.31.0",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "accelerate==1.0.1",
    "safetensors==0.5.3",
    "ftfy==6.2.3",
    "timm==1.0.15",
    "loguru==0.7.3",
    "scikit-learn",
    "datasets",
], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repo ready:", REPO)


Cloning into '/kaggle/working/cospy'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
urwid 5.0.1 requires wcwidth!=0.9.0,!=0.9.1,>=0.4, but you have wcwidth 0.2.14 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
prettytable 3.18.0 requires wcwidth>=0.3.5, but you have wcwidth 0.2.14 which is incompatible.


Repo ready: /kaggle/working/cospy


In [3]:
# ============================================================
# VERIFY / AUTO-DETECT SDXL FAKE DATA
# Robust to Kaggle dataset names / owners, e.g. sdxl-mscocoo
# ============================================================

from pathlib import Path
import os

EXT = {".jpg", ".jpeg", ".png", ".webp"}
DATASETS_ROOT = INPUT / "datasets"

def list_images(folder):
    return sorted(
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in EXT
    )

def mounted_dataset_roots():
    roots = []
    if not DATASETS_ROOT.exists():
        return roots

    for owner in DATASETS_ROOT.iterdir():
        if not owner.is_dir():
            continue

        for dataset in owner.iterdir():
            if dataset.is_dir():
                roots.append(dataset)

    return roots

MOUNTED_DATASETS = mounted_dataset_roots()

def find_sdxl_fake_dir(source):
    candidates = []

    for dataset_root in MOUNTED_DATASETS:
        synthetic_root = dataset_root / "SDXL-full" / "synthetic"
        source_dir = synthetic_root / source

        if not source_dir.is_dir():
            continue

        # Find generator folder by semantic content, not dataset mount name.
        model_candidates = [
            p for p in source_dir.iterdir()
            if p.is_dir()
            and (
                p.name == MODEL_NAME
                or "stable-diffusion-xl-base-1.0" in p.name.lower()
            )
        ]

        if not model_candidates:
            continue

        # Prefer exact generator name.
        model_candidates.sort(
            key=lambda p: (
                p.name != MODEL_NAME,
                len(str(p)),
            )
        )

        model_dir = model_candidates[0]
        candidates.append(
            (
                len(str(model_dir)),
                dataset_root,
                model_dir,
            )
        )

    assert candidates, (
        f"Không tìm thấy SDXL fake data cho source={source}.\n"
        f"Cần cấu trúc: <dataset>/SDXL-full/synthetic/{source}/"
        f"<stable-diffusion-xl-base-1.0>"
    )

    candidates.sort(key=lambda x: x[0])
    return candidates[0][1], candidates[0][2]


fake_dirs = {}
fake_by_source = {}

print("=" * 110)
print("SDXL FAKE DATA CHECK")
print("=" * 110)

for source in SOURCES:
    dataset_root, fake_dir = find_sdxl_fake_dir(source)
    imgs = list_images(fake_dir)

    fake_dirs[source] = fake_dir
    fake_by_source[source] = imgs

    print(
        f"{source:10s} | "
        f"{len(imgs):5d} images | "
        f"dataset={dataset_root.name} | "
        f"{fake_dir}"
    )

    assert len(imgs) >= NUM_PER_CLASS, (
        f"{source}: fake={len(imgs)}, cần >= {NUM_PER_CLASS}"
    )

print("=" * 110)
print("[PASS] All 5 SDXL fake sources are ready.")


SDXL FAKE DATA CHECK
mscoco     |  5000 images | dataset=sdxl-mscocoo | /kaggle/input/datasets/buivinh2311/sdxl-mscocoo/SDXL-full/synthetic/mscoco/stabilityai@stable-diffusion-xl-base-1.0
flickr     |  5000 images | dataset=sdxl-flickr | /kaggle/input/datasets/buivinh2311/sdxl-flickr/SDXL-full/synthetic/flickr/stabilityai@stable-diffusion-xl-base-1.0
cc3m       |  5000 images | dataset=sdxl-cc3m | /kaggle/input/datasets/buivinh2311/sdxl-cc3m/SDXL-full/synthetic/cc3m/stabilityai@stable-diffusion-xl-base-1.0
textcaps   |  5000 images | dataset=sdxl-textcaps | /kaggle/input/datasets/buivinh2311/sdxl-textcaps/SDXL-full/synthetic/textcaps/stabilityai@stable-diffusion-xl-base-1.0
sbu        |  5000 images | dataset=sdxl-sbu | /kaggle/input/datasets/buivinh2311/sdxl-sbu/SDXL-full/synthetic/sbu/stabilityai@stable-diffusion-xl-base-1.0
[PASS] All 5 SDXL fake sources are ready.


In [4]:
# ============================================================
# PREPARE REAL SOURCES
# ============================================================

from datasets import load_dataset
from tqdm.auto import tqdm
import requests
import zipfile

REAL_WORK = WORK / "cospy_real_sources"
REAL_ROOT = REAL_WORK / "real_image_examples"
REAL_WORK.mkdir(parents=True, exist_ok=True)

REAL_EXAMPLES_URL = (
    "https://huggingface.co/datasets/ruojiruoli/Co-Spy-Misc/"
    "resolve/main/real_image_examples.zip"
)

# ------------------------------------------------------------
# 1) Find Co-Spy_v1 regardless of owner / mount naming
# ------------------------------------------------------------

def find_cospy_v1_root():
    candidates = []

    for dataset_root in MOUNTED_DATASETS:
        name = dataset_root.name.lower()

        if (
            ("co-spy" in name or "cospy" in name)
            and "final" not in name
        ):
            candidates.append(dataset_root)

    assert candidates, "Không tìm thấy Co-Spy_v1 trong Kaggle Inputs."

    candidates.sort(
        key=lambda p: (
            p.name.lower() not in {"co-spy-v1", "co-spy_v1"},
            len(str(p)),
        )
    )

    return candidates[0]

COSPY_V1_ROOT = find_cospy_v1_root()

MSCOCO_REAL = (
    COSPY_V1_ROOT
    / "mscoco2017"
    / "mscoco2017"
    / "val2017"
)

assert MSCOCO_REAL.is_dir(), (
    f"Không tìm thấy MSCOCO val2017 tại: {MSCOCO_REAL}"
)

print("Co-Spy_v1 :", COSPY_V1_ROOT)
print("MSCOCO    :", MSCOCO_REAL)

# ------------------------------------------------------------
# 2) Official real examples: CC3M / SBU / TextCaps
# ------------------------------------------------------------

need_real_zip = not all(
    (REAL_ROOT / name).exists()
    for name in ["CC3M", "SBU", "TextCaps"]
)

if need_real_zip:
    zip_path = REAL_WORK / "real_image_examples.zip"

    if not zip_path.exists():
        print("\nDownloading real_image_examples.zip ...")

        with requests.get(
            REAL_EXAMPLES_URL,
            stream=True,
            timeout=(30, 180),
        ) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))

            with open(zip_path, "wb") as f, tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc="real examples",
            ) as bar:
                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

    print("Extracting real examples ...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(REAL_WORK)

    zip_path.unlink(missing_ok=True)
    print("Deleted real_image_examples.zip after extraction.")

# ------------------------------------------------------------
# 3) Flickr30k test — all 9 parquet shards
# ------------------------------------------------------------

print("\nLoading Flickr30k TEST Parquet ...")

FLICKR_TEST_SHARDS = [
    (
        "https://huggingface.co/datasets/nlphuji/flickr30k/resolve/"
        "7fa323c61061a3a9079b248b75c50947fa01b1ce/"
        f"TEST/test-{i:05d}-of-00009.parquet"
    )
    for i in range(9)
]

flickr_ds = load_dataset(
    "parquet",
    data_files={"test": FLICKR_TEST_SHARDS},
    split="test",
)

assert "image" in flickr_ds.column_names, (
    f"Flickr parquet thiếu cột image: {flickr_ds.column_names}"
)

assert len(flickr_ds) >= NUM_PER_CLASS, (
    f"Flickr test chỉ có {len(flickr_ds)} ảnh, cần {NUM_PER_CLASS}"
)

print(f"Flickr ready: {len(flickr_ds)} real images")

REAL_DIRS = {
    "mscoco": MSCOCO_REAL,
    "cc3m": REAL_ROOT / "CC3M",
    "textcaps": REAL_ROOT / "TextCaps",
    "sbu": REAL_ROOT / "SBU",
}

real_by_source = {}

print("\n" + "=" * 90)
print("REAL DATA CHECK")
print("=" * 90)

for source, folder in REAL_DIRS.items():
    assert folder.exists(), (
        f"Missing real source: {source} -> {folder}"
    )

    imgs = list_images(folder)
    real_by_source[source] = imgs

    print(
        f"{source:10s}: "
        f"{len(imgs):5d} real images | "
        f"{folder}"
    )

    assert len(imgs) >= NUM_PER_CLASS, (
        f"{source}: real={len(imgs)}, cần >= {NUM_PER_CLASS}"
    )

print(
    f"{'flickr':10s}: "
    f"{len(flickr_ds):5d} real images | "
    "Flickr30k TEST Parquet"
)

print("=" * 90)
print("[PASS] Real sources are ready.")


Co-Spy_v1 : /kaggle/input/datasets/datnguyentrung/co-spy-v1
MSCOCO    : /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/val2017



real examples:   0%|          | 0.00/885M [00:00<?, ?B/s]

Extracting real examples ...
Deleted real_image_examples.zip after extraction.

Loading Flickr30k TEST Parquet ...


TEST/test-00000-of-00009.parquet:   0%|          | 0.00/459M [00:00<?, ?B/s]

TEST/test-00001-of-00009.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

TEST/test-00002-of-00009.parquet:   0%|          | 0.00/474M [00:00<?, ?B/s]

... (4 outputs left out)

TEST/test-00007-of-00009.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

TEST/test-00008-of-00009.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Flickr ready: 31014 real images

REAL DATA CHECK
mscoco    :  5000 real images | /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/val2017
cc3m      :  2000 real images | /kaggle/working/cospy_real_sources/real_image_examples/CC3M
textcaps  :  2000 real images | /kaggle/working/cospy_real_sources/real_image_examples/TextCaps
sbu       :  2000 real images | /kaggle/working/cospy_real_sources/real_image_examples/SBU
flickr    : 31014 real images | Flickr30k TEST Parquet
[PASS] Real sources are ready.


In [5]:
# ============================================================
# RESOLVE NEW FULL FUSION V3 CHECKPOINT
# Only one checkpoint is required:
#   co-spy-fusion-v3/checkpoint_best.pth
# ============================================================

def find_full_fusion_v3_checkpoint():
    candidates = []

    for dataset_root in MOUNTED_DATASETS:
        ckpt = dataset_root / "checkpoint_best.pth"
        name = dataset_root.name.lower().replace("_", "-")

        if (
            ckpt.is_file()
            and "fusion" in name
            and "v3" in name
        ):
            candidates.append(ckpt)

    assert candidates, (
        "Không tìm thấy full Fusion V3 checkpoint. "
        "Cần Kaggle Input dạng: co-spy-fusion-v3/checkpoint_best.pth"
    )

    candidates.sort(
        key=lambda p: (
            "co-spy-fusion-v3" not in p.parent.name.lower().replace("_", "-"),
            len(str(p)),
        )
    )
    return candidates[0]


FUSION_CKPT = find_full_fusion_v3_checkpoint()

print("Full Fusion V3 checkpoint:")
print(" ", FUSION_CKPT)
print(" Size:", f"{FUSION_CKPT.stat().st_size / 1024**3:.3f} GB")
print("[PASS] New full Fusion V3 checkpoint resolved.")


Full Fusion V3 checkpoint:
  /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
 Size: 3.593 GB
[PASS] New full Fusion V3 checkpoint resolved.


In [6]:
# ============================================================
# INPUT PREFLIGHT SUMMARY
# ============================================================

print("=" * 100)
print("SDXL EVAL PREFLIGHT")
print("=" * 100)

for source in SOURCES:
    print(
        f"✅ fake {source:10s}: "
        f"{len(fake_by_source[source]):5d} | "
        f"{fake_dirs[source]}"
    )

print(f"✅ real mscoco   : {MSCOCO_REAL}")
print(f"✅ full Fusion V3: {FUSION_CKPT}")
print("=" * 100)
print("[PASS] Dataset + real data + new full checkpoint are all resolved.")


SDXL EVAL PREFLIGHT
✅ fake mscoco    :  5000 | /kaggle/input/datasets/buivinh2311/sdxl-mscocoo/SDXL-full/synthetic/mscoco/stabilityai@stable-diffusion-xl-base-1.0
✅ fake flickr    :  5000 | /kaggle/input/datasets/buivinh2311/sdxl-flickr/SDXL-full/synthetic/flickr/stabilityai@stable-diffusion-xl-base-1.0
✅ fake cc3m      :  5000 | /kaggle/input/datasets/buivinh2311/sdxl-cc3m/SDXL-full/synthetic/cc3m/stabilityai@stable-diffusion-xl-base-1.0
✅ fake textcaps  :  5000 | /kaggle/input/datasets/buivinh2311/sdxl-textcaps/SDXL-full/synthetic/textcaps/stabilityai@stable-diffusion-xl-base-1.0
✅ fake sbu       :  5000 | /kaggle/input/datasets/buivinh2311/sdxl-sbu/SDXL-full/synthetic/sbu/stabilityai@stable-diffusion-xl-base-1.0
✅ real mscoco   : /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/val2017
✅ full Fusion V3: /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
[PASS] Dataset + real data + new full checkpoint are all resolved.


In [7]:
# ============================================================
# EVAL HELPERS — FUSION V3
# ============================================================
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, accuracy_score
from tqdm.auto import tqdm

from evaluate import Detector
from utils import png_to_jpeg, bootstrap_ap_ci

device = "cuda:0" if torch.cuda.is_available() else "cpu"
assert device == "cuda:0", "Hãy bật GPU Accelerator trên Kaggle."
print("GPU:", torch.cuda.get_device_name(0))

class SourceImageDataset(Dataset):
    def __init__(self, source, num_per_class, sem_transform, artifact_transform):
        self.source = source
        self.n = num_per_class
        self.sem_transform = sem_transform
        self.artifact_transform = artifact_transform

        self.fake = fake_by_source[source][:self.n]

        if source == "flickr":
            self.real = None
            self.real_refs = [
                f"hf://nlphuji/flickr30k/test/{i}"
                for i in range(self.n)
            ]
        else:
            self.real = real_by_source[source][:self.n]
            self.real_refs = [str(p) for p in self.real]

        self.fake_refs = [str(p) for p in self.fake]
        self.refs = self.real_refs + self.fake_refs

    def __len__(self):
        return self.n * 2

    def _load_real(self, idx):
        if self.source == "flickr":
            return flickr_ds[idx]["image"].convert("RGB")
        return Image.open(self.real[idx]).convert("RGB")

    def __getitem__(self, idx):
        if idx < self.n:
            image = self._load_real(idx)
            label = 0
        else:
            image = Image.open(self.fake[idx - self.n]).convert("RGB")
            label = 1

        # Match official CoSpyBench behavior
        image = png_to_jpeg(image, quality=JPEG_QUALITY)

        sem_image = self.sem_transform(image)
        artifact_views = self.artifact_transform(image)

        return sem_image, artifact_views, label


def metrics_from_scores(scores, y_true):
    scores = np.asarray(scores, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = (scores >= 0.5).astype(np.int64)

    return {
        "size": int(len(y_true)),
        "num_real": int((y_true == 0).sum()),
        "num_fake": int((y_true == 1).sum()),
        "AP": float(average_precision_score(y_true, scores)),
        "AP_95CI": bootstrap_ap_ci(
            scores.tolist(),
            y_true.tolist(),
            n_samples=500,
            seed=1024,
        ),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Real_correct": float((scores[y_true == 0] < 0.5).mean()),
        "Fake_detected": float((scores[y_true == 1] >= 0.5).mean()),
    }


def eval_source_v3(model, source):
    ds = SourceImageDataset(
        source=source,
        num_per_class=NUM_PER_CLASS,
        sem_transform=model.sem.test_transform,
        artifact_transform=model.artifact_view_transform(ARTIFACT_VIEWS),
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    scores, labels_all = [], []
    t0 = time.time()

    with torch.inference_mode():
        for sem_images, artifact_views, labels in tqdm(
            loader,
            desc=f"Fusion v3 | {source}",
            unit="batch",
        ):
            # Keep raw artifact views as (B, V, C, H, W).
            # The runtime patch handles the encoded (B, V, F) representation
            # inside forward_features(), where it belongs.
            assert artifact_views.ndim == 5, (
                f"Expected raw artifact views (B,V,C,H,W), got "
                f"{tuple(artifact_views.shape)}"
            )

            pred = model.predict_views(sem_images, artifact_views)
            scores.extend(float(v) for v in pred)
            labels_all.extend(labels.tolist())

    result = metrics_from_scores(scores, labels_all)
    result["Runtime_minutes"] = float((time.time() - t0) / 60)

    return result, np.asarray(scores), np.asarray(labels_all), ds.refs

print("[PASS] Fusion v3 eval helpers ready.")


GPU: Tesla T4
[PASS] Fusion v3 eval helpers ready.


In [8]:
# ============================================================
# LOAD NEW FULL FUSION V3 CHECKPOINT
# The checkpoint already contains:
#   sem.* + art.* + npr.* + fusion_head.*
#
# CoSpyFusionDetector's constructor still expects three branch
# checkpoint files, so we create TEMP bootstrap branch checkpoints
# from this same full state_dict. After construction, the complete
# full state_dict is loaded again with strict=True.
# ============================================================

from detectors.sd_v1_4 import CoSpyFusionDetector

fusion_path = FUSION_CKPT
assert fusion_path.is_file(), f"Missing checkpoint: {fusion_path}"

print("Loading new full Fusion V3 checkpoint...")
print("Fusion v3:", fusion_path)

state = torch.load(
    fusion_path,
    map_location="cpu",
    weights_only=True,
)

assert isinstance(state, dict), type(state)

required_prefixes = [
    "sem.",
    "art.",
    "npr.",
    "fusion_head.",
]

for prefix in required_prefixes:
    assert any(k.startswith(prefix) for k in state), (
        f"Fusion checkpoint does not contain required prefix: {prefix}"
    )

print("state_dict keys:", len(state))
print("Has sem.*        :", any(k.startswith("sem.") for k in state))
print("Has art.*        :", any(k.startswith("art.") for k in state))
print("Has npr.*        :", any(k.startswith("npr.") for k in state))
print("Has fusion_head.*:", any(k.startswith("fusion_head.") for k in state))

# ------------------------------------------------------------
# Constructor bootstrap:
# extract the three submodule state_dicts from THIS SAME checkpoint.
# These files are temporary and are only needed because the current
# constructor loads branch weights before we can load the full model.
# ------------------------------------------------------------

BOOTSTRAP_DIR = WORK / "fusion_v3_bootstrap"
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)

def save_submodule_state(full_state, prefix, out_path):
    sub_state = {
        k[len(prefix):]: v
        for k, v in full_state.items()
        if k.startswith(prefix)
    }
    assert sub_state, f"No keys found for prefix: {prefix}"
    torch.save(sub_state, out_path)
    return out_path

semantic_path = save_submodule_state(
    state, "sem.", BOOTSTRAP_DIR / "semantic.pth"
)
artifact_path = save_submodule_state(
    state, "art.", BOOTSTRAP_DIR / "artifact.pth"
)
npr_path = save_submodule_state(
    state, "npr.", BOOTSTRAP_DIR / "npr.pth"
)

print("\nBuilding CO-SPY Fusion v3 runtime...")
print("Artifact bootstrap:", artifact_path)
print("Semantic bootstrap:", semantic_path)
print("NPR bootstrap     :", npr_path)

model = CoSpyFusionDetector(
    semantic_weights_path=str(semantic_path),
    artifact_weights_path=str(artifact_path),
    npr_weights_path=str(npr_path),
    fusion_version="v3",
    branch_dropout=0.15,
    artifact_views=ARTIFACT_VIEWS,
)

# Diagnostic compatibility check first.
incompat = model.load_state_dict(state, strict=False)

print("\nCompatibility report")
print("Missing keys   :", len(incompat.missing_keys))
print("Unexpected keys:", len(incompat.unexpected_keys))

if incompat.missing_keys:
    print("\nFirst missing keys:")
    for k in incompat.missing_keys[:30]:
        print(" ", k)

if incompat.unexpected_keys:
    print("\nFirst unexpected keys:")
    for k in incompat.unexpected_keys[:30]:
        print(" ", k)

assert not incompat.missing_keys, (
    "Checkpoint is missing model keys; stop before evaluation."
)
assert not incompat.unexpected_keys, (
    "Checkpoint contains unexpected model keys; stop before evaluation."
)

# Final strict load: this is the actual model used for evaluation.
model.load_state_dict(state, strict=True)

del state
gc.collect()

model.to(device)
model.eval()

print("\n[PASS] NEW full Fusion V3 checkpoint loaded successfully.")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

I0000 00:00:1795173049.445456      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading new full Fusion V3 checkpoint...
Fusion v3: /kaggle/input/datasets/datnguyentrung/co-spy-fusion-v3/checkpoint_best.pth
state_dict keys: 1232
Has sem.*        : True
Has art.*        : True
Has npr.*        : True
Has fusion_head.*: True

Building CO-SPY Fusion v3 runtime...
Artifact bootstrap: /kaggle/working/fusion_v3_bootstrap/artifact.pth
Semantic bootstrap: /kaggle/working/fusion_v3_bootstrap/semantic.pth
NPR bootstrap     : /kaggle/working/fusion_v3_bootstrap/npr.pth


open_clip_model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]


Compatibility report
Missing keys   : 0
Unexpected keys: 0

[PASS] NEW full Fusion V3 checkpoint loaded successfully.
Device: cuda:0
GPU   : Tesla T4


In [9]:
# ============================================================
# PATCH FUSION V3 EVAL RUNTIME
# Accept artifact feature views shaped (B, V, F)
# ============================================================

original_prepare_artifact_views = model._prepare_artifact_views

def patched_prepare_artifact_views(x_art):
    # During raw-image evaluation:
    # input artifact views are (B, V, C, H, W).
    # extract_features() converts them to (B, V, F).
    # The current runtime omitted this valid 3-D feature-view case.
    if x_art.ndim == 3:
        return x_art

    # Keep original handling for 2-D / 4-D / 5-D tensors.
    return original_prepare_artifact_views(x_art)

model._prepare_artifact_views = patched_prepare_artifact_views

print("[PATCH] Fusion v3 now accepts artifact feature views (B, V, F).")


[PATCH] Fusion v3 now accepts artifact feature views (B, V, F).


In [10]:
# ============================================================
# PREFLIGHT SDXL — TEST EXACTLY 1 BATCH BEFORE FULL EVAL
# ============================================================

test_source = "mscoco"

test_ds = SourceImageDataset(
    source=test_source,
    num_per_class=min(NUM_PER_CLASS, 32),
    sem_transform=model.sem.test_transform,
    artifact_transform=model.artifact_view_transform(ARTIFACT_VIEWS),
)

test_loader = DataLoader(
    test_ds,
    batch_size=min(BATCH, 16),
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

sem_images, artifact_views, labels = next(iter(test_loader))

print("Before normalization:")
print("  sem_images     :", tuple(sem_images.shape))
print("  artifact_views :", tuple(artifact_views.shape))
print("  labels         :", tuple(labels.shape))

assert artifact_views.ndim == 5, (
    f"Expected raw artifact views (B,V,C,H,W), got "
    f"{tuple(artifact_views.shape)}"
)

print("\nRaw artifact views are valid; testing end-to-end forward...")

with torch.inference_mode():
    probe_pred = model.predict_views(sem_images, artifact_views)

print("\nPrediction probe:")
print("  count :", len(probe_pred))
print("  first :", probe_pred[:5])

assert len(probe_pred) == len(labels)
assert all(0.0 <= float(x) <= 1.0 for x in probe_pred)

print("\n[PASS] One-batch Fusion v3 inference works.")


Before normalization:
  sem_images     : (16, 3, 384, 384)
  artifact_views : (16, 1, 3, 224, 224)
  labels         : (16,)

Raw artifact views are valid; testing end-to-end forward...

Prediction probe:
  count : 16
  first : [4.921063122970137e-12, 1.6352764475868753e-07, 2.1002800193459237e-12, 6.3146371812039526e-12, 3.798646645717177e-14]

[PASS] One-batch Fusion v3 inference works.


In [11]:
# ============================================================
# RUN FULL SDXL — 5 SOURCES
# ============================================================
results = {}
outputs = {}
all_scores = []
all_labels = []

total_start = time.time()

for source in SOURCES:
    r, scores, labels, refs = eval_source_v3(model, source)

    results[source] = r
    outputs[source] = {
        "scores": scores,
        "labels": labels,
        "refs": refs,
    }

    all_scores.append(scores)
    all_labels.append(labels)

    print(
        f"{source:8s} | "
        f"AP={r['AP']*100:6.2f}% | "
        f"Acc={r['Accuracy']*100:6.2f}% | "
        f"Real={r['Real_correct']*100:6.2f}% | "
        f"Fake={r['Fake_detected']*100:6.2f}% | "
        f"{r['Runtime_minutes']:.1f} min"
    )

all_scores = np.concatenate(all_scores)
all_labels = np.concatenate(all_labels)
all_pred = (all_scores >= 0.5).astype(int)

macro_ap = float(np.mean([results[s]["AP"] for s in SOURCES]))
macro_acc = float(np.mean([results[s]["Accuracy"] for s in SOURCES]))

pooled_ap = float(average_precision_score(all_labels, all_scores))
pooled_acc = float(accuracy_score(all_labels, all_pred))

summary = {
    "benchmark": "Co-Spy-Bench / SDXL",
    "generator": MODEL_NAME,
    "model": "CO-SPY Fusion v3",
    "architecture": "Semantic + Artifact + NPR -> Fusion v3",
    "num_per_class_per_source": NUM_PER_CLASS,
    "jpeg_quality": JPEG_QUALITY,
    "sources": SOURCES,
    "per_source": results,
    "Macro_AP": macro_ap,
    "Macro_Accuracy": macro_acc,
    "Pooled_AP": pooled_ap,
    "Pooled_Accuracy": pooled_acc,
    "Total_runtime_minutes": float((time.time() - total_start) / 60),
}

print("\n" + "=" * 76)
print("SDXL / CO-SPY FUSION V3")
print("=" * 76)

for source in SOURCES:
    print(
        f"{source:8s} | "
        f"AP {results[source]['AP']*100:6.2f}% | "
        f"Acc {results[source]['Accuracy']*100:6.2f}%"
    )

print("-" * 76)
print(f"Macro AP       : {macro_ap*100:.2f}%")
print(f"Macro Accuracy : {macro_acc*100:.2f}%")
print(f"Pooled AP      : {pooled_ap*100:.2f}%")
print(f"Pooled Accuracy: {pooled_acc*100:.2f}%")
print(f"Total runtime  : {summary['Total_runtime_minutes']:.1f} min")
print("=" * 76)


Fusion v3 | mscoco:   0%|          | 0/250 [00:00<?, ?batch/s]

mscoco   | AP= 99.27% | Acc= 61.52% | Real=100.00% | Fake= 23.05% | 38.0 min


Fusion v3 | flickr:   0%|          | 0/250 [00:00<?, ?batch/s]

flickr   | AP= 99.42% | Acc= 60.15% | Real=100.00% | Fake= 20.30% | 36.1 min


Fusion v3 | cc3m:   0%|          | 0/250 [00:00<?, ?batch/s]

cc3m     | AP= 67.05% | Acc= 55.55% | Real= 91.45% | Fake= 19.65% | 35.9 min


Fusion v3 | textcaps:   0%|          | 0/250 [00:00<?, ?batch/s]

textcaps | AP= 74.55% | Acc= 55.93% | Real= 96.20% | Fake= 15.65% | 36.5 min


Fusion v3 | sbu:   0%|          | 0/250 [00:00<?, ?batch/s]

sbu      | AP= 94.80% | Acc= 54.95% | Real= 99.85% | Fake= 10.05% | 35.9 min

SDXL / CO-SPY FUSION V3
mscoco   | AP  99.27% | Acc  61.52%
flickr   | AP  99.42% | Acc  60.15%
cc3m     | AP  67.05% | Acc  55.55%
textcaps | AP  74.55% | Acc  55.93%
sbu      | AP  94.80% | Acc  54.95%
----------------------------------------------------------------------------
Macro AP       : 87.02%
Macro Accuracy : 57.62%
Pooled AP      : 84.75%
Pooled Accuracy: 57.62%
Total runtime  : 182.4 min


In [12]:
# ============================================================
# SAVE RESULTS
# ============================================================

with open(OUT_DIR / "result.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

with open(OUT_DIR / "predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "source",
        "path_or_ref",
        "label",
        "fusion_v3_score",
        "predicted_label",
    ])

    for source in SOURCES:
        o = outputs[source]
        for ref, y, score in zip(
            o["refs"],
            o["labels"],
            o["scores"],
        ):
            w.writerow([
                source,
                ref,
                int(y),
                float(score),
                int(score >= 0.5),
            ])

print("SDXL evaluation saved:")
print(" ", OUT_DIR / "result.json")
print(" ", OUT_DIR / "predictions.csv")


SDXL evaluation saved:
  /kaggle/working/eval_sdxl_fusionv3/result.json
  /kaggle/working/eval_sdxl_fusionv3/predictions.csv
